# Практическая работа №2 —группировка и сравнение

## Задание 1. Загрузка данных

In [ ]:

from google.colab import files

uploaded = files.upload()

import os
file_name = next(iter(uploaded))
print("Загружен файл:", file_name)


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(file_name)
df.head()


## Задание 2. Знакомство с датасетом

In [ ]:

print("Количество строк:", df.shape[0])
print("Количество столбцов:", df.shape[1])


print("\nНазвания столбцов:")
print(df.columns.tolist())


print("\nТипы данных:")
print(df.dtypes)


## Задание 3. Анализ пассажиров

In [ ]:

print("Количество пассажиров по классам:")
print(df['Pclass'].value_counts().sort_index())


print("\nКоличество мужчин и женщин:")
print(df['Sex'].value_counts())


bins = [0, 12, 17, 59, 100]
labels = ['Дети (0–12)', 'Подростки (13–17)', 'Взрослые (18–59)', 'Пожилые (60+)']

df['AgeGroup'] = pd.cut(
    df['Age'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

print("\nКоличество пассажиров по возрастным группам:")
print(df['AgeGroup'].value_counts().sort_index())


print("\nСредний возраст:", round(df['Age'].mean(), 2))
print("Медианный возраст:", df['Age'].median())


print("Средняя стоимость билета:", round(df['Fare'].mean(), 2))


## Задание 4. Сравнение пассажиров разных классов

In [ ]:
mean_fare = df.groupby('Pclass')['Fare'].mean()
print("Средняя стоимость билета по классам:")
print(mean_fare.round(2))

mean_age = df.groupby('Pclass')['Age'].mean()
print("\nСредний возраст по классам:")
print(mean_age.round(2))

sex_by_class = pd.crosstab(df['Pclass'], df['Sex'])
print("\nКоличество мужчин и женщин в каждом классе:")
print(sex_by_class)

most_popular_class = df['Pclass'].value_counts().idxmax()
most_popular_count = df['Pclass'].value_counts().max()
print(f"\nБольше всего пассажиров было в {most_popular_class}-м классе: {most_popular_count}.")


In [ ]:

max_fare_row = df.loc[df['Fare'].idxmax(), ['Fare', 'Pclass', 'Age', 'Sex', 'Survived']]

print("Самый дорогой билет:")
print("Стоимость:", max_fare_row['Fare'])
print("Класс:", int(max_fare_row['Pclass']))
print("Возраст:", max_fare_row['Age'])
print("Пол:", max_fare_row['Sex'])
print("Выжил:", "Да" if max_fare_row['Survived'] == 1 else "Нет")


## Задание 5. Анализ выживаемости

In [ ]:

survival_counts = df['Survived'].value_counts().sort_index()

print("Погибло:", survival_counts.get(0, 0))
print("Выжило:", survival_counts.get(1, 0))

survival_by_sex = df.groupby('Sex')['Survived'].agg(['sum', 'count', 'mean'])
survival_by_sex.columns = ['Выжило', 'Всего', 'Доля выживших']

print("\nВыживаемость мужчин и женщин:")
print(survival_by_sex.round(3))

survival_by_class_sex = (
    df.groupby(['Pclass', 'Sex'])['Survived']
      .agg(['sum', 'count', 'mean'])
)
survival_by_class_sex.columns = ['Выжило', 'Всего', 'Доля выживших']

print("\nВыживаемость по классу и полу:")
print(survival_by_class_sex.round(3))

group_survival = df.groupby(['Pclass', 'Sex'])['Survived'].mean()
best_group = group_survival.idxmax()
best_rate = group_survival.max()

print(
    f"\nНаибольшая доля выживших: класс {best_group[0]}, "
    f"{'женщины' if best_group[1] == 'female' else 'мужчины'} — "
    f"{best_rate:.1%}."
)


## Задание 6. Визуализация

In [ ]:
class_counts = df['Pclass'].value_counts().sort_index()

plt.figure(figsize=(7, 5))
plt.bar(class_counts.index.astype(str), class_counts.values)
plt.title('Количество пассажиров по классам')
plt.xlabel('Класс')
plt.ylabel('Количество пассажиров')
plt.show()


In [ ]:
# График выживаемости по полу
survival_sex = df.groupby('Sex')['Survived'].mean()

plt.figure(figsize=(7, 5))
plt.bar(['Женщины', 'Мужчины'],
        [survival_sex.get('female', 0), survival_sex.get('male', 0)])
plt.title('Выживаемость по полу')
plt.xlabel('Пол')
plt.ylabel('Доля выживших')
plt.ylim(0, 1)
plt.show()
